# <font color="#418FDE" size="6.5" uppercase>**Popular Scikit-Learn ML Models**</font>

>Last update: 20260325.
    
By the end of this Lecture, you will be able to:
- Describe support vector machines, k-nearest neighbors, and decision trees for classification and regression problems. 
- Implement common scikit-learn ML models on prepared datasets to solve CE classification and regression tasks. 
- Evaluate model performance using suitable metrics, visualizations, and cross-validation to assess reliability and generalization. 


## **1. Classical ML Models**

### **1.1. Support Vector Machines**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/JHU/Artificial Intelligence in Civil Engineering Applications/Module_04/Lecture_B/image_01_01.jpg?v=1774483473" width="250">



>* SVMs maximize the margin between classes.
>* Support vectors define a robust boundary.

>* SVMs model linear and nonlinear patterns.
>* Margin-error tradeoff helps prevent overfitting.

>* SVM regression ignores small errors within margins.
>* Scaling and tuning are essential for performance.



In [ ]:
#@title Python Code - Support Vector Machines

# Support vector machines find wide separating boundaries.
# This example uses satellite land cover data.
# Such classes support civil site studies.

import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

# Download the training satellite file.
!wget -q -O ./sat.trn https://archive.ics.uci.edu/ml/machine-learning-databases/statlog/satimage/sat.trn
# Download the testing satellite file.
!wget -q -O ./sat.tst https://archive.ics.uci.edu/ml/machine-learning-databases/statlog/satimage/sat.tst

# Set a seed for reproducibility.
np.random.seed(42)
# Read space separated training data.
train_df = pd.read_csv("./sat.trn", sep="\s+", header=None)
# Read space separated testing data.
test_df = pd.read_csv("./sat.tst", sep="\s+", header=None)

# Split features and class labels.
X_train = train_df.iloc[:, :-1].to_numpy()
y_train = train_df.iloc[:, -1].to_numpy()
X_test = test_df.iloc[:, :-1].to_numpy()

y_test = test_df.iloc[:, -1].to_numpy()
# Check that feature counts match.
if X_train.shape[1] != X_test.shape[1]:
    raise ValueError("Training and testing features differ.")

# Build a scaled support vector classifier.
svm_model = Pipeline([
    ("scaler", StandardScaler()),
    ("svc", SVC(kernel="rbf", C=2.0, gamma="scale"))
])

# Train the classifier on training data.
svm_model.fit(X_train, y_train)
# Predict land cover classes.
y_pred = svm_model.predict(X_test)
# Measure classification accuracy.
acc = accuracy_score(y_test, y_pred)

# Print a short teaching summary.
print("Dataset:", X_train.shape[0], "train samples and", X_test.shape[0], "test samples")
print("Features per sample:", X_train.shape[1])
print("Test accuracy:", round(acc, 3))
print("SVM uses support vectors to separate classes")

print("Scaled features help because bands use different ranges")
print("Land cover classes can inform site and infrastructure studies")
print("Examples include soil, vegetation, and built surfaces")

# Plot a compact confusion matrix.
ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred,
    cmap="Blues",
    colorbar=False
)

# Add a clear plot title.
plt.title("Support Vector Machine on Landsat Classes")
# Keep the layout tidy.
plt.tight_layout()
plt.show()



### **1.2. Nearest Neighbor Basics**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/JHU/Artificial Intelligence in Civil Engineering Applications/Module_04/Lecture_B/image_01_02.jpg?v=1774483505" width="250">



>* Predictions use outcomes of similar past cases.
>* Classification votes; regression averages nearby examples.

>* Distance depends on scaling and similarity.
>* Neighbor count balances noise and oversmoothing.

>* Flexible for complex local patterns.
>* Needs scaled, representative, low-dimensional data.



In [ ]:
#@title Python Code - Nearest Neighbor Basics

# Nearest neighbors compare nearby feature patterns.
# This example uses satellite land classes.
# Similar samples guide each new prediction.

import numpy as np
import pandas as pd
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix
import matplotlib.pyplot as plt

# Set a seed for repeatability.
np.random.seed(7)

# Download the training data.
!wget -q -O ./sat.trn https://archive.ics.uci.edu/ml/machine-learning-databases/statlog/satimage/sat.trn

# Download the testing data.
!wget -q -O ./sat.tst https://archive.ics.uci.edu/ml/machine-learning-databases/statlog/satimage/sat.tst

# Read space separated text files.
train_df = pd.read_csv("./sat.trn", sep="\s+", header=None)
test_df = pd.read_csv("./sat.tst", sep="\s+", header=None)

# Split features and target labels.
X_train = train_df.iloc[:, :-1].copy()
y_train = train_df.iloc[:, -1].copy()

# Split testing features and labels.
X_test = test_df.iloc[:, :-1].copy()
y_test = test_df.iloc[:, -1].copy()

# Check basic dataset sizes.
assert X_train.shape[0] > 100, "Training data is too small."
assert X_train.shape[1] == X_test.shape[1], "Feature counts must match."

# Standardize features before distance calculations.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

# Apply the same scaling.
X_test_scaled = scaler.transform(X_test)

# Build a simple nearest neighbor classifier.
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_scaled, y_train)

# Predict the test labels.
y_pred = knn.predict(X_test_scaled)
acc = accuracy_score(y_test, y_pred)

# Inspect one test sample.
sample_index = 0
sample = X_test_scaled[sample_index:sample_index + 1]

# Find the nearest stored samples.
distances, indices = knn.kneighbors(sample, n_neighbors=5)
neighbor_labels = y_train.iloc[indices[0]].to_numpy()

# Predict the chosen sample.
sample_pred = knn.predict(sample)[0]
sample_true = y_test.iloc[sample_index]

# Print a short teaching summary.
print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)
print("Test accuracy:", round(acc, 3))
print("Chosen test sample index:", sample_index)

# Explain nearby samples in feature space.
print("Neighbor labels:", neighbor_labels.tolist())
print("Neighbor distances:", np.round(distances[0], 2).tolist())
print("Predicted class:", int(sample_pred), "True class:", int(sample_true))
print("KNN uses nearby scaled samples for voting.")

# Build a compact confusion matrix.
cm = confusion_matrix(y_test, y_pred)
classes = np.unique(y_test)

# Show one simple performance plot.
plt.figure(figsize=(6, 4))
plt.imshow(cm, cmap="Blues")
plt.title("KNN confusion matrix")
plt.xlabel("Predicted class")

# Finish the labeled plot.
plt.ylabel("True class")
plt.xticks(range(len(classes)), classes)
plt.yticks(range(len(classes)), classes)
plt.tight_layout()
plt.show()



### **1.3. Decision Tree Basics**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/JHU/Artificial Intelligence in Civil Engineering Applications/Module_04/Lecture_B/image_01_03.jpg?v=1774483536" width="250">



>* Flowchart model splits data using simple rules.
>* Handles varied inputs with interpretable nonlinear predictions.

>* Trees split data to improve group purity.
>* Splitting repeats until stopping rules apply.

>* Trees risk overfitting, underfitting, and instability.
>* They remain valuable for transparent predictions.



In [ ]:
#@title Python Code - Decision Tree Basics

# Decision trees split data using simple questions.
# This example predicts concrete compressive strength.
# Small plots help explain tree structure.

# Install Excel reader if needed.
# !pip install -q xlrd

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeRegressor, plot_tree
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score

# Set a seed for repeatable results.
np.random.seed(42)

# Download the concrete dataset file.
!wget -q -O ./concrete.xls "https://archive.ics.uci.edu/ml/machine-learning-databases/concrete/compressive/Concrete_Data.xls"

# Load the spreadsheet into pandas.
df = pd.read_excel("./concrete.xls")

# Show the dataset size briefly.
print("Rows and columns:", df.shape)

# Separate inputs and target column.
X = df.iloc[:, :-1]
y = df.iloc[:, -1]

# Split data into training and testing parts.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Build a small tree for clarity.
model = DecisionTreeRegressor(
    max_depth=3, random_state=42
)

# Train the decision tree regressor.
model.fit(X_train, y_train)

# Predict compressive strength on test data.
y_pred = model.predict(X_test)

# Measure prediction accuracy simply.
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

# Report a few key results.
print("Test MAE:", round(mae, 2), "MPa")
print("Test R^2:", round(r2, 2))
print("Tree depth:", model.get_depth())
print("Leaf nodes:", model.get_n_leaves())

# Explain the first split simply.
root_feature = X.columns[model.tree_.feature[0]]
root_value = model.tree_.threshold[0]
print("First question uses:", root_feature)
print("Split threshold:", round(root_value, 2))

# Plot a small readable decision tree.
plt.figure(figsize=(10, 6))
plot_tree(
    model,
    feature_names=X.columns,
    filled=True,
    rounded=True,
    fontsize=8
)

# Add a clear plot title.
plt.title("Decision Tree for Concrete Strength")
plt.tight_layout()
plt.show()



## **2. Applying Models**

### **2.1. Classification Model Application**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/JHU/Artificial Intelligence in Civil Engineering Applications/Module_04/Lecture_B/image_02_01.jpg?v=1774483564" width="250">



>* Prepare labeled data and split train-test.
>* Train classifier and evaluate test predictions.

>* Classification models suit different data patterns.
>* Scaling, settings, and interpretation matter.

>* Match predictions to engineering decisions and risks.
>* Use balanced data and expert-guided workflows.



In [ ]:
#@title Python Code - Classification Model Application

# This example applies classification models.
# We compare KNN and SVM.
# The data represent land cover.

# Download the training dataset.
!wget -q -O ./sat.trn https://archive.ics.uci.edu/ml/machine-learning-databases/statlog/satimage/sat.trn

# Download the testing dataset.
!wget -q -O ./sat.tst https://archive.ics.uci.edu/ml/machine-learning-databases/statlog/satimage/sat.tst

# Import beginner friendly libraries.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Import scikit-learn tools.
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, ConfusionMatrixDisplay

# Set a deterministic random seed.
np.random.seed(42)
# Load the downloaded text files.
train_df = pd.read_csv("sat.trn", sep="\s+", header=None)
test_df = pd.read_csv("sat.tst", sep="\s+", header=None)

# Split features and target labels.
X_train = train_df.iloc[:, :-1].copy()
y_train = train_df.iloc[:, -1].copy()
X_test = test_df.iloc[:, :-1].copy()
y_test = test_df.iloc[:, -1].copy()

# Check that shapes look valid.
if X_train.shape[1] != X_test.shape[1]:
    raise ValueError("Training and testing features must match.")
# Scale features for distance models.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Fit a KNN classifier.
knn_model = KNeighborsClassifier(n_neighbors=5)
knn_model.fit(X_train_scaled, y_train)
knn_pred = knn_model.predict(X_test_scaled)

# Fit an SVM classifier.
svm_model = SVC(kernel="rbf", C=2.0, gamma="scale")
svm_model.fit(X_train_scaled, y_train)
svm_pred = svm_model.predict(X_test_scaled)

# Compute simple accuracy scores.
knn_acc = accuracy_score(y_test, knn_pred)
svm_acc = accuracy_score(y_test, svm_pred)
# Count prediction agreement.
agreement = np.mean(knn_pred == svm_pred)

# Print a short summary.
print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)
print("Number of land classes:", y_train.nunique())
print("KNN accuracy:", round(knn_acc, 3))
print("SVM accuracy:", round(svm_acc, 3))
print("Model agreement:", round(agreement, 3))

# Plot one confusion matrix.
ConfusionMatrixDisplay.from_predictions(
    y_test,
    svm_pred,
    cmap="Blues",
    colorbar=False,
)

# Add a clear plot title.
plt.title("SVM confusion matrix for land-cover classes")
plt.tight_layout()
plt.show()



### **2.2. Regression Model Application**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/JHU/Artificial Intelligence in Civil Engineering Applications/Module_04/Lecture_B/image_02_02.jpg?v=1774483598" width="250">



>* Regression predicts continuous civil engineering outcomes.
>* Model choice depends on data patterns.

>* Use train-test splits to check generalization.
>* Choose models based on data patterns.

>* Compare predictions with reality for decisions.
>* Use caution beyond training data range.



In [ ]:
#@title Python Code - Regression Model Application

# This example applies regression to concrete data.
# Students compare two scikit-learn regression models.
# Results show prediction quality for strength.

# Install Excel reader if needed.
# !pip install -q xlrd

# Download the concrete strength dataset.
!wget -q -O ./concrete.xls "https://archive.ics.uci.edu/ml/machine-learning-databases/concrete/compressive/Concrete_Data.xls"

# Import beginner-friendly analysis libraries.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Import scikit-learn modeling tools.
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR

# Import tree model and metrics.
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, r2_score

# Set a fixed random seed.
np.random.seed(42)

# Read the downloaded Excel file.
df = pd.read_excel("./concrete.xls")

# Separate inputs and target column.
X = df.iloc[:, :-1]
y = df.iloc[:, -1]

# Split data into training and testing sets.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Build a scaled support vector regressor.
svr_model = make_pipeline(
    StandardScaler(), SVR(kernel="rbf", C=20, epsilon=1.0)
)

# Build a decision tree regressor.
tree_model = DecisionTreeRegressor(
    max_depth=6, random_state=42
)

# Train both regression models.
svr_model.fit(X_train, y_train)
tree_model.fit(X_train, y_train)

# Predict concrete strength on test data.
svr_pred = svr_model.predict(X_test)
tree_pred = tree_model.predict(X_test)

# Calculate simple evaluation metrics.
svr_mae = mean_absolute_error(y_test, svr_pred)
svr_r2 = r2_score(y_test, svr_pred)
tree_mae = mean_absolute_error(y_test, tree_pred)

tree_r2 = r2_score(y_test, tree_pred)

# Print a short summary.
print("Dataset shape:", df.shape)
print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])
print("SVR MAE:", round(svr_mae, 2), "MPa")

print("SVR R^2:", round(svr_r2, 3))
print("Tree MAE:", round(tree_mae, 2), "MPa")
print("Tree R^2:", round(tree_r2, 3))

# Choose the better model by R squared.
if svr_r2 >= tree_r2:
    better_name = "SVR"
    better_pred = svr_pred
else:
    better_name = "Decision Tree"
    better_pred = tree_pred

# Plot observed versus predicted strengths.
plt.figure(figsize=(6, 5))
plt.scatter(y_test, better_pred, alpha=0.7, color="teal")
plt.xlabel("Observed strength (MPa)")
plt.ylabel("Predicted strength (MPa)")

# Add a reference agreement line.
low_value = min(y_test.min(), better_pred.min())
high_value = max(y_test.max(), better_pred.max())
plt.plot([low_value, high_value], [low_value, high_value], "r--")

# Finish the comparison figure.
plt.title("Best model: " + better_name)
plt.tight_layout()
plt.show()



### **2.3. Training on Civil Data**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/JHU/Artificial Intelligence in Civil Engineering Applications/Module_04/Lecture_B/image_02_03.jpg?v=1774483636" width="250">



>* Civil data comes from real systems.
>* Model quality depends on representative training data.

>* Split data; scale features when needed.
>* Train SVM, KNN, trees on examples.

>* Use data matching real operating conditions.
>* Apply domain knowledge for reliable generalization.



In [ ]:
#@title Python Code - Training on Civil Data

# This example trains on civil concrete data.
# We build one simple regression pipeline.
# Each step is explained briefly below.

# Import notebook shell support.
from IPython import get_ipython

# Import core data libraries.
import numpy as np
import pandas as pd

# Import plotting and sklearn tools.
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

# Import preprocessing and model tools.
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Import regression model and metrics.
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error, r2_score

# Set a seed for repeatability.
np.random.seed(42)

# Download the concrete dataset quietly.
get_ipython().system('wget -q -O ./concrete.xls "https://archive.ics.uci.edu/ml/machine-learning-databases/concrete/compressive/Concrete_Data.xls"')

# Load the spreadsheet into pandas.
df = pd.read_excel("./concrete.xls")

# Show dataset size only.
print("Rows and columns:", df.shape)

# Separate inputs from target values.
X = df.iloc[:, :-1]
y = df.iloc[:, -1]

# Split data into training and testing parts.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Confirm the split sizes.
print("Training rows:", X_train.shape[0])
print("Testing rows:", X_test.shape[0])

# Build a pipeline with scaling and SVR.
model = Pipeline([
    ("scaler", StandardScaler()),
    ("svr", SVR(kernel="rbf", C=10.0, epsilon=2.0))
])

# Train the model on civil data.
model.fit(X_train, y_train)

# Predict concrete strength on test data.
y_pred = model.predict(X_test)

# Measure prediction quality simply.
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

# Print short beginner friendly results.
print("Model: scaled support vector regression")
print("Target: concrete compressive strength")
print("Mean absolute error:", round(mae, 2), "MPa")
print("R^2 score:", round(r2, 3))

# Plot predicted versus actual strengths.
plt.figure(figsize=(5, 5))
plt.scatter(y_test, y_pred, alpha=0.7)

# Add a simple reference line.
low = min(y_test.min(), y_pred.min())
high = max(y_test.max(), y_pred.max())

# Draw the ideal prediction line.
plt.plot([low, high], [low, high], color="red")
plt.xlabel("Actual strength (MPa)")

# Label the vertical axis clearly.
plt.ylabel("Predicted strength (MPa)")
plt.title("Concrete strength predictions")

# Show the single required plot.
plt.tight_layout()
plt.show()



## **3. Model Evaluation Methods**

### **3.1. Evaluation Metrics**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/JHU/Artificial Intelligence in Civil Engineering Applications/Module_04/Lecture_B/image_03_01.jpg?v=1774483668" width="250">



>* Classification metrics beyond accuracy often matter.
>* Regression metrics should match error consequences.

>* Choose metrics based on error consequences.
>* Use multiple metrics; one score misleads.

>* Compare training, testing, and baseline performance.
>* Interpret errors for real-world reliability decisions.



In [ ]:
#@title Python Code - Evaluation Metrics

# This script shows simple evaluation metrics.
# We use regression and classification examples.
# The datasets come from UCI sources.

# Install xlrd if spreadsheet reading fails.
# !pip install xlrd

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score
from sklearn.tree import DecisionTreeRegressor

from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import r2_score

from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import ConfusionMatrixDisplay

# Set a seed for repeatability.
np.random.seed(42)

# Download the concrete regression dataset.
!wget -q -O ./concrete.xls "https://archive.ics.uci.edu/ml/machine-learning-databases/concrete/compressive/Concrete_Data.xls"

# Download the satellite classification files.
!wget -q -O ./sat.trn "https://archive.ics.uci.edu/ml/machine-learning-databases/statlog/satimage/sat.trn"
!wget -q -O ./sat.tst "https://archive.ics.uci.edu/ml/machine-learning-databases/statlog/satimage/sat.tst"

# Read the concrete spreadsheet.
concrete = pd.read_excel("./concrete.xls")

# Split features and target values.
X_reg = concrete.iloc[:, :-1]
y_reg = concrete.iloc[:, -1]

# Create train and test sets.
Xr_train, Xr_test, yr_train, yr_test = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)

# Train a simple regression model.
reg_model = DecisionTreeRegressor(max_depth=5, random_state=42)
reg_model.fit(Xr_train, yr_train)

# Predict concrete strength values.
yr_pred = reg_model.predict(Xr_test)
mae_value = mean_absolute_error(yr_test, yr_pred)
r2_value = r2_score(yr_test, yr_pred)

# Estimate reliability with cross validation.
cv_r2 = cross_val_score(
    reg_model, X_reg, y_reg, cv=5, scoring="r2"
)

# Read the satellite training data.
sat_train = pd.read_csv("./sat.trn", sep="\s+", header=None)

# Read the satellite testing data.
sat_test = pd.read_csv("./sat.tst", sep="\s+", header=None)

# Split classification features and labels.
Xc_train = sat_train.iloc[:, :-1]
yc_train = sat_train.iloc[:, -1]

# Split testing features and labels.
Xc_test = sat_test.iloc[:, :-1]
yc_test = sat_test.iloc[:, -1]

# Use a small beginner friendly classifier.
clf_model = KNeighborsClassifier(n_neighbors=5)
clf_model.fit(Xc_train, yc_train)

# Predict land cover classes.
yc_pred = clf_model.predict(Xc_test)
acc_value = accuracy_score(yc_test, yc_pred)
cm = confusion_matrix(yc_test, yc_pred)

# Print a short metric summary.
print("Regression MAE:", round(mae_value, 2))
print("Regression R2:", round(r2_value, 3))
print("CV mean R2:", round(cv_r2.mean(), 3))
print("Classification accuracy:", round(acc_value, 3))
print("Confusion matrix shape:", cm.shape)

# Plot the confusion matrix once.
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(confusion_matrix=cm).plot(ax=ax, colorbar=False)

# Add a clear title.
ax.set_title("Satellite Classification Confusion Matrix")
plt.tight_layout()
plt.show()



### **3.2. Model Result Visualization**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/JHU/Artificial Intelligence in Civil Engineering Applications/Module_04/Lecture_B/image_03_02.jpg?v=1774483702" width="250">



>* Visuals reveal errors hidden by metrics.
>* Confusion matrices expose imbalance and critical misclassifications.

>* Scatter plots reveal regression error patterns.
>* Residual plots expose nonlinearity, outliers, and heterogeneity.

>* Curves reveal fit, complexity, and generalization.
>* Visual stability builds trust in engineering models.



In [ ]:
#@title Python Code - Model Result Visualization

# This example visualizes regression model results.
# It uses concrete strength engineering data.
# Simple plots help explain prediction errors.

# Download the dataset file quietly.
!wget -q -O ./concrete.xls "https://archive.ics.uci.edu/ml/machine-learning-databases/concrete/compressive/Concrete_Data.xls"

# Import beginner friendly data tools.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Import simple sklearn model tools.
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

# Set a seed for repeatability.
np.random.seed(42)

# Load the spreadsheet into pandas.
df = pd.read_excel("./concrete.xls")

# Separate inputs and target column.
X = df.iloc[:, :-1]
y = df.iloc[:, -1]

# Check dataset size before training.
if len(df) < 20:
    raise ValueError("Dataset is unexpectedly too small.")

# Split data into train and test.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Fit a simple linear regression model.
model = LinearRegression()
model.fit(X_train, y_train)

# Predict concrete strength on test data.
y_pred = model.predict(X_test)
residuals = y_test - y_pred

# Calculate two easy evaluation metrics.
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

# Print short interpretation friendly results.
print("Rows and columns:", df.shape)
print("Test MAE in MPa:", round(mae, 2))
print("Test R^2 score:", round(r2, 3))
print("Top plot: predicted versus actual values.")
print("Bottom plot: residuals around zero are better.")

# Create one figure with two subplots.
fig, axes = plt.subplots(1, 2, figsize=(8, 4))

# Plot predicted values against actual values.
axes[0].scatter(y_test, y_pred, alpha=0.7, color="steelblue")
axes[0].plot([y.min(), y.max()], [y.min(), y.max()], color="red")
axes[0].set_title("Predicted vs Actual")
axes[0].set_xlabel("Actual strength, MPa")

# Finish labels for the first plot.
axes[0].set_ylabel("Predicted strength, MPa")

# Plot residuals to inspect error patterns.
axes[1].scatter(y_pred, residuals, alpha=0.7, color="darkorange")
axes[1].axhline(0, color="red")
axes[1].set_title("Residual Plot")
axes[1].set_xlabel("Predicted strength, MPa")

# Finish labels and show the figure.
axes[1].set_ylabel("Residual, actual minus predicted")
plt.tight_layout()
plt.show()



### **3.3. Cross Validation Reliability**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/JHU/Artificial Intelligence in Civil Engineering Applications/Module_04/Lecture_B/image_03_03.jpg?v=1774483730" width="250">



>* Repeated splits better estimate unseen performance.
>* Shows consistency across varied, noisy data.

>* Score consistency across folds shows model stability.
>* Supports fair model comparison for classification and regression.

>* Match validation to real data structure.
>* Improves model selection and detects overfitting.



In [ ]:
#@title Python Code - Cross Validation Reliability

# Cross validation checks model reliability.
# Concrete strength offers civil engineering context.
# This example uses simple regression tools.

# Download the concrete dataset file.
!wget -q -O ./concrete.xls "https://archive.ics.uci.edu/ml/machine-learning-databases/concrete/compressive/Concrete_Data.xls"

# Import beginner friendly libraries.
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold, cross_val_score, train_test_split

# Import a simple regression model.
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsRegressor

# Set a fixed random seed.
np.random.seed(42)

# Load the spreadsheet into pandas.
df = pd.read_excel("./concrete.xls")

# Separate inputs and target column.
X = df.iloc[:, :-1]
y = df.iloc[:, -1]

# Check the dataset dimensions.
if len(df) < 50 or X.shape[1] != 8:
    raise ValueError("Dataset shape is not expected.")

# Build a simple scaled KNN model.
model = make_pipeline(
    StandardScaler(),
    KNeighborsRegressor(n_neighbors=5)
)

# Make one train test split.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Fit the model once.
model.fit(X_train, y_train)

# Score the single split.
single_r2 = model.score(X_test, y_test)

# Define five fold cross validation.
kfold = KFold(n_splits=5, shuffle=True, random_state=42)

# Compute repeated validation scores.
cv_scores = cross_val_score(
    model, X, y, cv=kfold, scoring="r2"
)

# Summarize reliability across folds.
cv_mean = cv_scores.mean()
cv_std = cv_scores.std()

# Print short teaching messages.
print("Rows and columns:", df.shape)
print("Single split R^2:", round(single_r2, 3))
print("Five fold mean R^2:", round(cv_mean, 3))
print("Five fold score spread:", round(cv_std, 3))

# Explain why cross validation helps.
print("Cross validation tests several data splits.")
print("Similar fold scores suggest stable performance.")
print("Large variation warns reliability may be weaker.")

# Plot fold scores for quick comparison.
import matplotlib.pyplot as plt
plt.figure(figsize=(6, 4))
plt.plot(range(1, 6), cv_scores, marker="o")

# Label the reliability plot clearly.
plt.axhline(cv_mean, color="red", linestyle="--")
plt.xticks(range(1, 6))
plt.xlabel("Fold number")
plt.ylabel("R^2 score")

# Finish the plot neatly.
plt.title("Concrete strength cross validation reliability")
plt.tight_layout()
plt.show()



# <font color="#418FDE" size="6.5" uppercase>**Popular Scikit-Learn ML Models**</font>


In this lecture, you learned to:
- Describe support vector machines, k-nearest neighbors, and decision trees for classification and regression problems. 
- Implement common scikit-learn ML models on prepared datasets to solve CE classification and regression tasks. 
- Evaluate model performance using suitable metrics, visualizations, and cross-validation to assess reliability and generalization. 

In the next Module (Module 5), we will go over 'Neural Networks, Part I'